In [ ]:
!pip install langchain
!pip install langchain-community
!pip install langchain-huggingface
!pip install langchain-groq
!pip install faiss-cpu
!pip install sentence-transformers
!pip install pypdf
!pip install gradio

print("All libraries installed!")

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
import gradio as gr
import os

print("All libraries imported successfully")

In [ ]:
from getpass import getpass

GROQ_API_KEY = getpass("Paste your Groq API key here: ")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("API key set")

In [ ]:
from google.colab import files

print("Please select one or more PDF files to upload...")
uploaded = files.upload()

pdf_files = [name for name in uploaded.keys() if name.lower().endswith(".pdf")]

if not pdf_files:
    print("No PDF files detected. Please re-run this cell and upload a .pdf file.")
else:
    print(f"\n {len(pdf_files)} PDF(s) uploaded successfully:")
    for f in pdf_files:
        print(f"   📄 {f}")

In [ ]:
all_pages = []

for pdf_file in pdf_files:
    try:
        loader = PyPDFLoader(pdf_file)
        pages = loader.load()
        all_pages.extend(pages)
        print(f" {pdf_file} — {len(pages)} pages loaded")
    except Exception as e:
        print(f" Failed to load {pdf_file}: {e}")
        print("   Tip: This usually means the PDF is scanned/image-based and has no extractable text.")

if not all_pages:
    print("\n No text could be extracted from any PDF. Cannot continue.")
else:
    print(f"\n📄 Total pages across all PDFs: {len(all_pages)}")

    # Show a preview from the first page to confirm text extracted correctly
    preview = all_pages[0].page_content[:300].strip()
    print(f"\n Preview of page 1:\n{preview}")

    # ── Chunking ──
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=100
    )
    chunks = text_splitter.split_documents(all_pages)

    print(f"\n Total chunks created: {len(chunks)}")
    print(f"\n Preview of Chunk 1:\n{chunks[0].page_content}")
    print(f"\n Preview of Chunk 2 (notice the overlap with Chunk 1):\n{chunks[1].page_content}")

In [ ]:
# Load the embedding model

# We use 'all-MiniLM-L6-v2' from HuggingFace:

print("Loading embedding model (downloads ~90MB on first run)...")

embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

print("Embedding model loaded!")

# Sanity check
test_vector = embedding_model.embed_query("What is machine learning?")
print(f"Vector size: {len(test_vector)} numbers per sentence")
print(f"First 5 values: {[round(v, 4) for v in test_vector[:5]]}")

In [ ]:
# Build the FAISS vector store

print(f"Embedding {len(chunks)} chunks and building FAISS index...")

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print("FAISS vector store built!")

test_results = vector_store.similarity_search(
    "What is this document about?",
    k=2
)

print("\nTest retrieval — top 2 chunks for 'What is this document about?'\n")
for i, doc in enumerate(test_results):
    print(f"--- Chunk {i+1} (Page {doc.metadata.get('page', 'N/A')}) ---")
    print(doc.page_content[:300])
    print()

In [ ]:
# Set up Groq's Llama 3 as the LLM

llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0.2,

    max_tokens=1024
)

print(" Groq LLM (Llama 3 8B) initialized!")

# Quick test
from langchain_core.messages import HumanMessage
test_response = llm.invoke([HumanMessage(content="Reply with just: API connection successful")])
print(f"\n API test: {test_response.content}")

In [ ]:
# Build the RAG pipeline

retriever = vector_store.as_retriever(
    search_kwargs={"k": 4}  # retrieve top 4 chunks
                            # more chunks = more context, but also more tokens used
)

# Define the prompt template
# {context} will be filled with the retrieved chunks
# {question} will be filled with the user's question
prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are a helpful assistant that answers questions strictly based on the provided document context.

Rules:
- Answer ONLY using information from the context below.
- If the answer is not clearly found in the context, say: "I couldn't find that information in the document."
- Be concise and clear. Do not repeat the question.
- If the context contains partial information, share what you found and note that it may be incomplete.

Context:
{context}

Question: {question}

Answer:"""
)

# The main RAG function — called every time user asks a question
def ask_question(question: str):
    """Run the full RAG pipeline: retrieve → augment → generate."""

    # Step 1: Retrieve the most relevant chunks from FAISS
    relevant_chunks = retriever.invoke(question)

    # Step 2: Combine chunk texts into one context block
    context = "\n\n".join([
        f"[Page {doc.metadata.get('page', '?')+1}]\n{doc.page_content}"
        for doc in relevant_chunks
    ])

    # Step 3: Fill the prompt template with context + question
    filled_prompt = prompt_template.format(
        context=context,
        question=question
    )

    # Step 4: Send to Groq API and get a response
    from langchain_core.messages import HumanMessage
    response = llm.invoke([HumanMessage(content=filled_prompt)])

    return response.content, relevant_chunks

print("RAG pipeline ready!")
print("\nRunning end-to-end test...")

# End-to-end test
test_answer, test_sources = ask_question("What is this document about?")

print(f"\nTest Answer:\n{test_answer}")
print(f"\nRetrieved {len(test_sources)} source chunks")

In [ ]:
# Launch the Gradio chat interface

import gradio as gr

def chat_with_pdf(message: str, history: list) -> str:
    """Gradio calls this function each time the user submits a message."""

    # Handle empty input gracefully
    if not message.strip():
        return "Please type a question."

    try:
        # Run the full RAG pipeline
        answer, sources = ask_question(message)

        # Format the response: answer + source citations
        response = f"**Answer:**\n{answer}\n\n---\n**Sources from document:**\n"

        for i, doc in enumerate(sources):
            page_num = doc.metadata.get('page', '?')
            # Show page number and first 200 chars of the chunk
            snippet = doc.page_content[:200].replace('\n', ' ').strip()
            response += f"\n**[{i+1}] Page {page_num + 1}:** {snippet}...\n"

        return response

    except Exception as e:
        # Surface any errors to the user instead of silent failure
        return f"Error!!: {str(e)}\n\nIf this is a rate limit error, wait a moment and try again."


# Get the names of uploaded files for the description
file_names = ", ".join(pdf_files)

demo = gr.ChatInterface(
    fn=chat_with_pdf,
    title="📚 PDF RAG Chatbot",
    description=f"Powered by **Groq (Llama 3)** + **FAISS** + **LangChain** | Document(s): `{file_names}`",
    examples=[
        "What is this document about?",
        "Summarize the key points.",
        "What are the main topics covered?",
        "What conclusions does the document reach?"
    ],
    theme=gr.themes.Soft()
)

print("Launching chatbot...")
demo.launch(share=True)